# PDF Processing Simulation

Este notebook ejecuta una simulación completa del flujo de procesamiento de PDF para verificar si la función `payroll.pdf_processor` detecta correctamente el nombre del empleado y procesa las fechas/horas.

In [ ]:
import pandas as pd
from payroll.pdf_processor import (
    analizar_estructura_pdf,
    extraer_datos_segun_estructura,
    procesar_datos_inteligente,
    convertir_a_dataframe_estandar,
    validar_datos_pdf,
    filtrar_registros_sin_asistencia,
    detectar_registros_incompletos,
    detectar_horarios_ambiguos,
)
from app import _empleado_coincide
from payroll.models import Employee

## Load Simulation File and Define PDF Text

Usamos el mismo texto simulado que está en `temp_pdf_simulation.py` y preparamos las líneas para el parser.

In [ ]:
texto_pdf = """
ID de persona    Nombre    Otra
15    Paz    Otro texto
2026-06-01 08:00
2026-06-01 17:00
2026-06-02 08:10
2026-06-02 17:05
"""
lineas = [l for l in texto_pdf.split("\n") if l.strip()]
lineas

## Analyze PDF Structure

Verificamos cómo el parser detecta la estructura del PDF simulado.

In [ ]:
estructura = analizar_estructura_pdf(lineas)
estructura

## Extract Raw Data from Simulated PDF

Convertimos las líneas en registros estructurados usando la estructura detectada.

In [ ]:
datos_brutos = extraer_datos_segun_estructura(lineas, estructura)
datos_brutos

## Process and Standardize the Extracted Data

Aplicamos el procesador inteligente y convertimos a DataFrame estándar.

In [ ]:
datos_procesados = procesar_datos_inteligente(datos_brutos)
df = convertir_a_dataframe_estandar(datos_procesados)
df

## Validate the Standardized PDF Data

Evaluamos si el DataFrame cumple con las reglas del parser y muestra errores.

In [ ]:
valido, errores = validar_datos_pdf(df)
valido, errores

## Test Employee Matching Logic

Probamos `_empleado_coincide` con un empleado `Paz Tatiana` y verificamos si las filas coinciden.

In [ ]:
empleado_prueba = Employee()
empleado_prueba.nombre = 'Paz Tatiana'
coincidencias = df['Empleado'].astype(str).apply(lambda x: _empleado_coincide(x, empleado_prueba))
coincidencias.tolist(), df[coincidencias]

## Detect Missing Attendance and Anomalies

Revisamos registros sin asistencia, incompletos y ambiguos.

In [ ]:
con_asistencia, sin_asistencia = filtrar_registros_sin_asistencia(df)
incompletos = detectar_registros_incompletos(con_asistencia)
ambiguos = detectar_horarios_ambiguos(con_asistencia)
con_asistencia.shape, sin_asistencia.shape, incompletos, ambiguos

## Inspect Output and Fix Any Processing Issues

Si el flujo falla, aquí revisaremos los resultados y ajustaremos el parser.